# NB1 — Data & Topology Prep — TopoDistil++

**Pipeline stage:** offline (CPU-only), one-time.

This notebook performs everything from Section 3.2–3.4 and 4.1 of the paper that does **not** touch the neural network:

1. Load PatchCamelyon (PCam) and build the fixed WSI-level splits.
2. Class-balanced subsample (~40–60k patches) from `train` only.
3. Nuclei extraction — H&E color deconvolution → threshold → connected components.
4. Persistent homology (H₀, H₁) on nuclei centroids via `ripser`.
5. Gaussian prior maps `A_gauss^{H0}`, `A_gauss^{H1}` + per-patch persistence entropy `H_i`.
6. **Robustness check** — StarDist cross-check on a held-out subset (Section 3.2).
7. Package everything into a checkpoint bundle to be added as an **input dataset** to NB2 (RepViT battery) and NB3 (MobileNetV4 cross-check).

> No labels are used anywhere in this notebook except for building the class-balanced sample and reporting the split. Topology computation is fully unsupervised (Section 3.3).

**Output of this notebook** (after you click *Save Version*) should be added to NB2/NB3 via *File → Add Input → Notebook Output*.


## 0. Setup

In [ ]:

!pip install -q ripser persim scikit-image opencv-python-headless h5py tqdm
# StarDist is only needed for the robustness check in Section 6.
!pip install -q stardist csbdeep


In [ ]:

import os, json, pickle, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import h5py
import cv2
from tqdm.auto import tqdm

from skimage.color import rgb2hed
from skimage.filters import threshold_otsu
from skimage.measure import label, regionprops
from skimage.morphology import remove_small_objects, binary_opening, disk
from skimage.feature import peak_local_max
from skimage.segmentation import watershed

from scipy import ndimage as ndi
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.spatial.distance import pdist, squareform
from scipy.stats import pearsonr

from ripser import ripser
from persim import bottleneck

warnings.filterwarnings("ignore")
np.random.seed(0)


## 1. Config

In [ ]:

CONFIG = {
    # --- Input: point this at wherever PCam's h5 files land after you attach
    # the Kaggle dataset (e.g. "andrewmvd/metastatic-tissue-classification-patchcamelyon").
    # We auto-discover matching filenames below, but override PCAM_DIR if discovery fails.
    "PCAM_DIR": "/kaggle/input/datasets/andrewmvd/metastatic-tissue-classification-patchcamelyon",

    # --- Subsampling ---
    "N_SUBSAMPLE_TRAIN": 100,   # class-balanced patches drawn from train split only
    "SEED": 0,

    # --- Nuclei extraction ---
    "MIN_NUCLEUS_AREA_PX": 8,      # drop connected components smaller than this (noise)
    "WATERSHED_MIN_DISTANCE": 2,   # min px separation between watershed seed peaks (splits touching nuclei)
    "OTSU_OFFSET": 0.0,            # additive offset applied to the Otsu threshold if needed

    # --- Persistent homology / topology maps ---
    "PATCH_SIZE": 96,              # native PCam patch size
    "SIGMA": 6.0,                  # Gaussian kernel width (pixels) for A_gauss
    "MAP_DOWNSAMPLE": 1,           # 1 = full 96x96 maps; use 2 to save space (48x48) if needed

    # --- StarDist robustness check (Section 3.2) ---
    "N_STARDIST_CHECK": 100,      # held-out patches for the segmentation cross-check
    "STARDIST_MODEL": "2D_versatile_he",

    # --- Output ---
    "OUT_DIR": "/kaggle/working/topology_checkpoint",
}

os.makedirs(CONFIG["OUT_DIR"], exist_ok=True)
np.random.seed(CONFIG["SEED"])
CONFIG


## 2. Load PCam and build the fixed split

PCam ships as six HDF5 files (`camelyonpatch_level_2_split_{train,valid,test}_{x,y}.h5`).
We auto-discover them under `/kaggle/input` — if your attached dataset uses different
filenames, just fix `PCAM_DIR` / the glob patterns below.

The official WSI-level split is preserved as-is (no slide's patches cross splits, since
that partition comes from the dataset itself) — we only *subsample* within `train`.


In [ ]:
from pathlib import Path

def find_pcam_files(root):
    root = Path(root)

    found = {}

    # Image HDF5 files in this Kaggle dataset
    image_names = {
        "train_x": "training_split.h5",
        "valid_x": "validation_split.h5",
        "test_x": "test_split.h5",
    }

    # Label HDF5 files
    label_names = {
        "train_y": "camelyonpatch_level_2_split_train_y.h5",
        "valid_y": "camelyonpatch_level_2_split_valid_y.h5",
        "test_y": "camelyonpatch_level_2_split_test_y.h5",
    }

    all_h5 = list(root.rglob("*.h5"))

    for key, filename in {**image_names, **label_names}.items():
        matches = [f for f in all_h5 if f.name == filename]
        if matches:
            found[key] = matches[0]

    return found


pcam_files = find_pcam_files(CONFIG["PCAM_DIR"])

required = [
    "train_x", "train_y",
    "valid_x", "valid_y",
    "test_x", "test_y"
]

missing = [k for k in required if k not in pcam_files]

if missing:
    print("Could not find:", missing)
    print("\nH5 files found:")
    for f in sorted(Path(CONFIG["PCAM_DIR"]).rglob("*.h5")):
        print(" ", f)
else:
    print("Found all PCam files:")
    for k, v in pcam_files.items():
        print(f"  {k}: {v}")

In [ ]:

def load_pcam_split(split):
    with h5py.File(pcam_files[f"{split}_x"], "r") as f:
        x_key = list(f.keys())[0]
        x = f[x_key]
        n = x.shape[0]
    with h5py.File(pcam_files[f"{split}_y"], "r") as f:
        y_key = list(f.keys())[0]
        y = np.array(f[y_key]).reshape(-1)
    return n, y

n_train, y_train_full = load_pcam_split("train")
n_valid, y_valid_full = load_pcam_split("valid")
n_test,  y_test_full  = load_pcam_split("test")

print(f"train: {n_train} patches | valid: {n_valid} | test: {n_test}")
print(f"train label balance: {y_train_full.mean():.3f} positive")


### 2.1 Class-balanced subsample from `train` only

In [ ]:

def class_balanced_subsample(y, n_total, seed):
    rng = np.random.RandomState(seed)
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]
    n_per_class = n_total // 2
    n_per_class = min(n_per_class, len(pos_idx), len(neg_idx))
    sel_pos = rng.choice(pos_idx, n_per_class, replace=False)
    sel_neg = rng.choice(neg_idx, n_per_class, replace=False)
    sel = np.concatenate([sel_pos, sel_neg])
    rng.shuffle(sel)
    return sel

train_sample_idx = class_balanced_subsample(
    y_train_full, CONFIG["N_SUBSAMPLE_TRAIN"], CONFIG["SEED"]
)
print(f"Subsampled {len(train_sample_idx)} train patches "
      f"({y_train_full[train_sample_idx].mean():.3f} positive)")

splits = {
    "train_subsample_idx": train_sample_idx.tolist(),
    "train_subsample_labels": y_train_full[train_sample_idx].tolist(),
    "valid_labels": y_valid_full.tolist(),
    "test_labels": y_test_full.tolist(),
    "n_valid": int(n_valid),
    "n_test": int(n_test),
    "seed": CONFIG["SEED"],
}
with open(f'{CONFIG["OUT_DIR"]}/splits.json', "w") as f:
    json.dump(splits, f)
print("Saved splits.json")


## 3. Nuclei extraction (Section 3.2)

Lightweight, CPU-only, unsupervised: H&E color deconvolution → Otsu threshold on the
hematoxylin (nuclei) channel → morphological cleanup → **distance-transform watershed
split** → connected components → centroids.

> **Fix (see Section 6 robustness check):** plain connected-component labeling merges
> touching/overlapping nuclei into a single blob, which undercounts nuclei in dense
> regions and biases the downstream H₀/H₁ topology. We add a distance-transform +
> local-maxima seeded watershed step to split touching nuclei before labeling.


In [ ]:
def extract_nuclei_centroids(
    patch_rgb,
    min_area=CONFIG["MIN_NUCLEUS_AREA_PX"]
):
    """Returns an (N, 2) array of (x, y) nuclei centroids for one RGB patch.

    Uses a distance-transform + local-maxima seeded watershed to split touching or
    overlapping nuclei that would otherwise be merged into a single connected-component
    blob (and therefore undercounted) -- see the StarDist robustness check in Section 6.
    """
    
    hed = rgb2hed(patch_rgb)
    h_channel = hed[:, :, 0]  # hematoxylin ~ nuclei stain
    h_channel = (
        (h_channel - h_channel.min())
        / (np.ptp(h_channel) + 1e-8)
    )

    try:
        thresh = threshold_otsu(
            h_channel
        ) + CONFIG["OTSU_OFFSET"]
    except ValueError:
        return np.zeros((0, 2))

    mask = h_channel > thresh
    mask = binary_opening(mask, footprint=disk(1))
    mask = remove_small_objects(
        mask,
        min_size=min_area
    )

    if not mask.any():
        return np.zeros((0, 2))

    # --- Watershed split: separate touching/overlapping nuclei ---
    # Distance-transform peaks act as one seed per nucleus; watershed then grows
    # each seed within the mask, splitting merged blobs back into individual nuclei.
    distance = ndi.distance_transform_edt(mask)
    coords = peak_local_max(
        distance,
        min_distance=CONFIG["WATERSHED_MIN_DISTANCE"],
        labels=mask,
    )
    peak_mask = np.zeros_like(distance, dtype=bool)
    peak_mask[tuple(coords.T)] = True
    markers = label(peak_mask)
    lbl = watershed(-distance, markers, mask=mask)

    props = regionprops(lbl)

    centroids = np.array([
        [p.centroid[1], p.centroid[0]]
        for p in props
    ])  # (x, y)

    return centroids


def load_patch(split, idx):
    with h5py.File(pcam_files[f"{split}_x"], "r") as f:
        x_key = list(f.keys())[0]
        return np.array(f[x_key][idx])

In [ ]:

# Quick visual sanity check on a handful of patches before running the full batch.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, idx in enumerate(train_sample_idx[:4]):
    patch = load_patch("train", idx)
    centroids = extract_nuclei_centroids(patch)
    axes[0, i].imshow(patch); axes[0, i].set_title(f"patch {idx}"); axes[0, i].axis("off")
    axes[1, i].imshow(patch)
    if len(centroids):
        axes[1, i].scatter(centroids[:, 0], centroids[:, 1], s=6, c="lime")
    axes[1, i].set_title(f"{len(centroids)} nuclei"); axes[1, i].axis("off")
plt.tight_layout(); plt.show()


## 4. Persistent homology (Section 3.3)

We compute H₀ (connected components → nuclei clustering) and H₁ (loops → architectural
disruption) via `ripser`. For each feature we also need a **critical-point coordinate**
so the Gaussian prior (Section 3.4) can be placed spatially:

- **H₀:** every point starts alive at birth 0 in a Vietoris–Rips filtration; the death of
  an H₀ feature is exactly a merge event of the minimum spanning tree (MST) of the point
  cloud. We compute the MST directly (`scipy.sparse.csgraph`) and take each edge's
  **midpoint** as that feature's critical-point location — this is exact, not an
  approximation.
- **H₁:** `ripser` gives birth/death radii but not generator coordinates directly. We
  approximate each H₁ feature's critical point as the midpoint of the closest point-pair
  whose pairwise distance matches its birth radius. This is a standard, clearly-flagged
  heuristic — good enough for a spatial *prior*, since Step 2 (Section 3.4) lets the
  network learn a residual correction on top of it.


In [ ]:

def compute_h0_persistence(centroids):
    """Exact H0 via MST: returns (births, deaths, crit_xy) with births all 0."""
    n = len(centroids)
    if n < 2:
        return np.zeros(0), np.zeros(0), np.zeros((0, 2))
    dmat = squareform(pdist(centroids))
    mst = minimum_spanning_tree(csr_matrix(dmat))
    mst = mst.tocoo()
    deaths = mst.data
    crit_xy = (centroids[mst.row] + centroids[mst.col]) / 2.0
    births = np.zeros_like(deaths)
    return births, deaths, crit_xy


def compute_h1_persistence(centroids, dmat=None):
    """H1 via ripser; critical point approximated by nearest-matching edge midpoint."""
    n = len(centroids)
    if n < 3:
        return np.zeros(0), np.zeros(0), np.zeros((0, 2))
    result = ripser(centroids, maxdim=1)
    dgm1 = result["dgms"][1]
    dgm1 = dgm1[np.isfinite(dgm1[:, 1])]  # drop features that never die (rare, boundary effect)
    if len(dgm1) == 0:
        return np.zeros(0), np.zeros(0), np.zeros((0, 2))

    if dmat is None:
        dmat = squareform(pdist(centroids))
    iu = np.triu_indices(n, k=1)
    pair_dists = dmat[iu]

    crit_xy = np.zeros((len(dgm1), 2))
    for k, (b, d) in enumerate(dgm1):
        j = np.argmin(np.abs(pair_dists - b))
        p, q = iu[0][j], iu[1][j]
        crit_xy[k] = (centroids[p] + centroids[q]) / 2.0

    return dgm1[:, 0], dgm1[:, 1], crit_xy


def persistence_entropy(persistences):
    """H_i = -sum (p_i / P) log(p_i / P), P = sum p_i."""
    persistences = persistences[persistences > 0]
    if len(persistences) == 0:
        return 0.0
    P = persistences.sum()
    probs = persistences / P
    return float(-(probs * np.log(probs + 1e-12)).sum())


## 5. Gaussian prior maps `A_gauss` (Section 3.4, Step 1)

Fixed, non-trainable spatial map built from persistence-weighted Gaussians at each
feature's critical point. `A_gauss^{H0}` and `A_gauss^{H1}` are computed and stored as
**separate channels** — H₀ and H₁ are treated as separable inputs throughout (Section 3.3).
The learned residual `f_θ` (Section 3.4, Step 2) is a *trainable* module and belongs in
NB2/NB3, not here.


In [ ]:

def gaussian_prior_map(crit_xy, persistences, size, sigma):
    A = np.zeros((size, size), dtype=np.float32)
    if len(crit_xy) == 0:
        return A
    yy, xx = np.mgrid[0:size, 0:size]
    for (x, y), p in zip(crit_xy, persistences):
        if p <= 0:
            continue
        d2 = (xx - x) ** 2 + (yy - y) ** 2
        A += p * np.exp(-d2 / (2 * sigma ** 2))
    return A


def process_patch(patch_rgb, size=CONFIG["PATCH_SIZE"], sigma=CONFIG["SIGMA"]):
    """Full per-patch pipeline: nuclei -> PH -> A_gauss maps -> entropy."""
    centroids = extract_nuclei_centroids(patch_rgb)
    n_nuclei = len(centroids)

    if n_nuclei < 3:
        empty_map = np.zeros((size, size), dtype=np.float32)
        return {
            "n_nuclei": n_nuclei,
            "h0": np.zeros((0, 2)), "h1": np.zeros((0, 2)),
            "A_h0": empty_map, "A_h1": empty_map,
            "entropy_h0": 0.0, "entropy_h1": 0.0,
        }

    dmat = squareform(pdist(centroids))
    b0, d0, xy0 = compute_h0_persistence(centroids)
    b1, d1, xy1 = compute_h1_persistence(centroids, dmat=dmat)

    p0 = d0 - b0
    p1 = d1 - b1

    A_h0 = gaussian_prior_map(xy0, p0, size, sigma)
    A_h1 = gaussian_prior_map(xy1, p1, size, sigma)

    return {
        "n_nuclei": n_nuclei,
        "h0": np.stack([b0, d0], axis=1) if len(b0) else np.zeros((0, 2)),
        "h1": np.stack([b1, d1], axis=1) if len(b1) else np.zeros((0, 2)),
        "A_h0": A_h0.astype(np.float16),
        "A_h1": A_h1.astype(np.float16),
        "entropy_h0": persistence_entropy(p0),
        "entropy_h1": persistence_entropy(p1),
    }


## 6. Batch run over the subsampled train set

Writes incrementally to disk (HDF5 for dense maps, pickle for variable-length persistence
diagrams) so a Kaggle session timeout doesn't lose completed work — re-running this cell
skips patches already present in the output file.


In [ ]:

h5_path = f'{CONFIG["OUT_DIR"]}/A_gauss_maps.h5'
diagrams_path = f'{CONFIG["OUT_DIR"]}/persistence_diagrams.pkl'
meta_path = f'{CONFIG["OUT_DIR"]}/patch_meta.csv'

# Resume support: load whatever's already been computed.
if os.path.exists(diagrams_path):
    with open(diagrams_path, "rb") as f:
        diagrams_store = pickle.load(f)
else:
    diagrams_store = {}

meta_rows = []
size = CONFIG["PATCH_SIZE"]

with h5py.File(h5_path, "a") as h5f:
    if "A_gauss" not in h5f:
        h5f.create_dataset(
            "A_gauss",
            shape=(len(train_sample_idx), 2, size, size),
            dtype="float16",
            chunks=(1, 2, size, size),
        )
        h5f.create_dataset("patch_idx", data=np.array(train_sample_idx))

    ds = h5f["A_gauss"]

    for pos, idx in enumerate(tqdm(train_sample_idx, desc="Processing patches")):
        if int(idx) in diagrams_store:
            continue  # already done in a previous session

        patch = load_patch("train", int(idx))
        result = process_patch(patch)

        ds[pos, 0] = result["A_h0"]
        ds[pos, 1] = result["A_h1"]

        diagrams_store[int(idx)] = {"h0": result["h0"], "h1": result["h1"]}

        meta_rows.append({
            "patch_idx": int(idx),
            "label": int(y_train_full[idx]),
            "n_nuclei": result["n_nuclei"],
            "entropy_h0": result["entropy_h0"],
            "entropy_h1": result["entropy_h1"],
        })

        if pos % 5000 == 0 and pos > 0:
            with open(diagrams_path, "wb") as f:
                pickle.dump(diagrams_store, f)

# Final flush
with open(diagrams_path, "wb") as f:
    pickle.dump(diagrams_store, f)

if meta_rows:
    meta_df = pd.DataFrame(meta_rows)
    if os.path.exists(meta_path):
        meta_df = pd.concat([pd.read_csv(meta_path), meta_df], ignore_index=True)
    meta_df.to_csv(meta_path, index=False)

print(f"Done. {len(diagrams_store)} patches processed.")
print(f"A_gauss maps -> {h5_path}")
print(f"Persistence diagrams -> {diagrams_path}")
print(f"Per-patch metadata -> {meta_path}")


## 7. Robustness check — StarDist cross-check (Section 3.2)

Re-run nuclei extraction on a held-out subset using a pretrained StarDist segmenter and
compare against the classical pipeline:

- **(a)** nuclei-count correlation between the two pipelines,
- **(b)** persistence-diagram similarity (bottleneck distance) between PH computed on
  classical-pipeline centroids vs. StarDist centroids.

High agreement on both confirms the classical pipeline is a faithful, non-artifactual
proxy — this check is reported once and does not require re-running full model training.


In [ ]:

from stardist.models import StarDist2D
from csbdeep.utils import normalize

stardist_model = StarDist2D.from_pretrained(CONFIG["STARDIST_MODEL"])

rng = np.random.RandomState(CONFIG["SEED"] + 1)
robustness_idx = rng.choice(train_sample_idx, CONFIG["N_STARDIST_CHECK"], replace=False)


In [ ]:
def stardist_centroids(patch_rgb, model, upscale=4, prob_thresh=0.3, nms_thresh=0.3):
    """Detect nuclei with a pretrained StarDist model.

    PCam patches are only 96x96px, well below the resolution '2D_versatile_he' was
    trained on -- at native scale, nuclei are a few pixels wide and fall below the
    model's default prob_thresh (0.692), so predict_instances silently returns zero
    detections rather than erroring. Fix: upscale before inference, lower prob_thresh
    to compensate, then rescale detected centroids back into the original 96x96
    coordinate frame so they stay aligned with the classical pipeline's output.
    """
    patch_uint8 = patch_rgb
    if patch_uint8.dtype != np.uint8:
        if patch_uint8.max() <= 1.0:
            patch_uint8 = (patch_uint8 * 255).astype(np.uint8)
        else:
            patch_uint8 = patch_uint8.astype(np.uint8)

    h, w = patch_uint8.shape[:2]
    if upscale != 1:
        patch_big = cv2.resize(patch_uint8, (w * upscale, h * upscale),
                                interpolation=cv2.INTER_CUBIC)
    else:
        patch_big = patch_uint8

    img = normalize(patch_big, 1, 99.8, axis=(0, 1))
    labels, _ = model.predict_instances(
        img, prob_thresh=prob_thresh, nms_thresh=nms_thresh, verbose=False
    )

    props = regionprops(labels)
    centroids = np.array([[p.centroid[1], p.centroid[0]] for p in props])
    if len(centroids) and upscale != 1:
        centroids = centroids / upscale  # back to original 96x96 coordinates
    return centroids


In [ ]:
# Sanity check: confirm the fix actually detects nuclei before running the full loop.
idx = 196642
patch = load_patch("train", idx)

classical_c = extract_nuclei_centroids(patch)
stardist_c = stardist_centroids(patch, stardist_model)  # uses the upscale+prob_thresh fix above

print("Patch:", idx)
print("Classical nuclei:", len(classical_c))
print("StarDist nuclei (fixed):", len(stardist_c))

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(patch); axes[0].set_title(f"Classical: {len(classical_c)}")
if len(classical_c):
    axes[0].scatter(classical_c[:, 0], classical_c[:, 1], s=6, c="lime")
axes[1].imshow(patch); axes[1].set_title(f"StarDist: {len(stardist_c)}")
if len(stardist_c):
    axes[1].scatter(stardist_c[:, 0], stardist_c[:, 1], s=6, c="cyan")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()


In [ ]:
idx = 196642

patch = load_patch("train", idx)

classical_c = extract_nuclei_centroids(patch)
stardist_c = stardist_centroids(patch, stardist_model)

print("Patch:", idx)
print("Classical nuclei:", len(classical_c))
print("StarDist nuclei:", len(stardist_c))

In [ ]:

classical_counts, stardist_counts = [], []
bottleneck_h0, bottleneck_h1 = [], []

for idx in tqdm(robustness_idx, desc="StarDist robustness check"):
    patch = load_patch("train", int(idx))

    classical_c = extract_nuclei_centroids(patch)
    stardist_c = stardist_centroids(patch, stardist_model)

    classical_counts.append(len(classical_c))
    stardist_counts.append(len(stardist_c))

    if len(classical_c) >= 3 and len(stardist_c) >= 3:
        _, d0_c, _ = compute_h0_persistence(classical_c)
        _, d1_c, xy1_c = compute_h1_persistence(classical_c)
        _, d0_s, _ = compute_h0_persistence(stardist_c)
        _, d1_s, xy1_s = compute_h1_persistence(stardist_c)

        dgm0_c = np.stack([np.zeros_like(d0_c), d0_c], axis=1) if len(d0_c) else np.zeros((0, 2))
        dgm0_s = np.stack([np.zeros_like(d0_s), d0_s], axis=1) if len(d0_s) else np.zeros((0, 2))
        dgm1_c = np.stack([np.zeros(len(d1_c)), d1_c], axis=1) if len(d1_c) else np.zeros((0, 2))
        dgm1_s = np.stack([np.zeros(len(d1_s)), d1_s], axis=1) if len(d1_s) else np.zeros((0, 2))

        if len(dgm0_c) and len(dgm0_s):
            bottleneck_h0.append(bottleneck(dgm0_c, dgm0_s))
        if len(dgm1_c) and len(dgm1_s):
            bottleneck_h1.append(bottleneck(dgm1_c, dgm1_s))

classical_counts = np.array(classical_counts)
stardist_counts = np.array(stardist_counts)

corr, pval = pearsonr(classical_counts, stardist_counts)

robustness_metrics = {
    "n_patches_checked": len(robustness_idx),
    "nuclei_count_pearson_r": float(corr),
    "nuclei_count_pearson_p": float(pval),
    "mean_classical_count": float(classical_counts.mean()),
    "mean_stardist_count": float(stardist_counts.mean()),
    "bottleneck_h0_mean": float(np.mean(bottleneck_h0)) if bottleneck_h0 else None,
    "bottleneck_h0_std": float(np.std(bottleneck_h0)) if bottleneck_h0 else None,
    "bottleneck_h1_mean": float(np.mean(bottleneck_h1)) if bottleneck_h1 else None,
    "bottleneck_h1_std": float(np.std(bottleneck_h1)) if bottleneck_h1 else None,
}

with open(f'{CONFIG["OUT_DIR"]}/stardist_robustness_metrics.json', "w") as f:
    json.dump(robustness_metrics, f, indent=2)

robustness_metrics


In [ ]:

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.scatter(classical_counts, stardist_counts, s=8, alpha=0.4)
lim = max(classical_counts.max(), stardist_counts.max())
ax.plot([0, lim], [0, lim], "r--", lw=1)
ax.set_xlabel("Classical pipeline nuclei count")
ax.set_ylabel("StarDist nuclei count")
ax.set_title(f"r = {robustness_metrics['nuclei_count_pearson_r']:.3f}")
plt.tight_layout(); plt.show()


## 8. Package checkpoint bundle for NB2 / NB3

In [ ]:

config_to_save = dict(CONFIG)
with open(f'{CONFIG["OUT_DIR"]}/config_used.json', "w") as f:
    json.dump(config_to_save, f, indent=2)

print("Checkpoint bundle contents:")
for f in sorted(Path(CONFIG["OUT_DIR"]).iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:35s} {size_mb:10.2f} MB")

print()
print("Next step: click 'Save Version' on this notebook.")
print("In NB2 and NB3, use File > Add Input > Notebook Output and select this")
print("notebook's latest output version to access everything under")
print(f'  /kaggle/input/<this-notebook-slug>/{Path(CONFIG["OUT_DIR"]).name}/')


## Output manifest (for reference in NB2/NB3)

| File | Contents |
|---|---|
| `splits.json` | Train-subsample indices + labels, valid/test labels, seed |
| `patch_meta.csv` | Per-patch: label, nuclei count, H₀/H₁ entropy |
| `persistence_diagrams.pkl` | `{patch_idx: {"h0": [[birth,death],...], "h1": [...]}}` |
| `A_gauss_maps.h5` | Dataset `A_gauss`, shape `(N, 2, 96, 96)` — channel 0 = H₀ map, channel 1 = H₁ map, `float16`; `patch_idx` dataset gives the row order |
| `stardist_robustness_metrics.json` | Section 3.2 cross-check results |
| `config_used.json` | Exact config this run used |

NB2/NB3 load `A_gauss` per-batch by index, build `f_θ`'s learned residual (Section 3.4,
Step 2) on top of it, and train the gating mechanism — no PH computation happens again
downstream.
